# FreshCart ELT — Snowflake
**Student:** Aayushi Rajesh Thakre | **Roll:** 23BCS004

This notebook is a runbook for the FreshCart SQL implementation. It executes the SQL files included in `sql/` against the user's Snowflake account. The SQL and observed results are based on the user's submitted FreshCart scripts and verification evidence.

> Set credentials through Colab secrets or environment variables. Never commit account passwords or private keys.


## 1. Install the Snowflake connector

In [ ]:
%pip -q install snowflake-connector-python

## 2. Connect to Snowflake
Set `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_PASSWORD`, `SNOWFLAKE_WAREHOUSE`, `SNOWFLAKE_ROLE` and `SNOWFLAKE_DATABASE` as environment variables or Colab secrets.

In [ ]:
import os
import snowflake.connector
from getpass import getpass

account = os.environ.get("SNOWFLAKE_ACCOUNT") or input("Snowflake account identifier: ")
user = os.environ.get("SNOWFLAKE_USER") or input("Snowflake username: ")
password = os.environ.get("SNOWFLAKE_PASSWORD") or getpass("Snowflake password: ")
warehouse = os.environ.get("SNOWFLAKE_WAREHOUSE") or input("Warehouse: ")
role = os.environ.get("SNOWFLAKE_ROLE") or input("Role (optional): ")

conn_args = dict(account=account, user=user, password=password, warehouse=warehouse)
if role:
    conn_args["role"] = role
conn = snowflake.connector.connect(**conn_args)
cur = conn.cursor()
print("Connected to Snowflake")


## 3. Run order
Run the setup script first. For the baseline, run Steps 11–15 once. If the baseline data needs correction, run the corrected rebuild script **instead of rerunning the Step 11 insert**, because that insert is append-only. Finally run Steps 16–20.

The scripts are stored beside this notebook in `sql/`. Upload the project folder to Colab or update `SQL_DIR` to its location.


In [ ]:
from pathlib import Path

SQL_DIR = Path("sql")

def run_sql_file(path):
    sql_text = Path(path).read_text(encoding="utf-8")
    results = cur.execute_string(sql_text, remove_comments=True)
    for i, result_cursor in enumerate(results, start=1):
        try:
            rows = result_cursor.fetchall()
            if rows:
                print(f"Statement {i}:")
                for row in rows[:20]:
                    print(row)
        except snowflake.connector.errors.ProgrammingError:
            pass
    conn.commit()
    print(f"Completed: {path}")

# Execute individually in the documented order:
# run_sql_file(SQL_DIR / "01_setup_tables_steps_06_10.sql")
# run_sql_file(SQL_DIR / "02_load_and_transform_steps_11_15.sql")
# For a corrected baseline, run the corrected rebuild instead of repeating step 11:
# run_sql_file(SQL_DIR / "03_corrected_rebuild_steps_12_15.sql")
# run_sql_file(SQL_DIR / "04_scd2_gold_quality_steps_16_20.sql")


## 4. Verification queries
These queries reproduce the principal row-count and Gold checks recorded in the evidence. Execute after the corresponding SQL steps.


In [ ]:
verification_queries = {
    "Bronze rows": "SELECT COUNT(*) FROM FRESHCART_DB.BRONZE.ORDERS_RAW",
    "Silver checked": "SELECT COUNT(*) FROM FRESHCART_DB.SILVER.ORDERS_CHECKED",
    "Silver clean": "SELECT COUNT(*) FROM FRESHCART_DB.SILVER.ORDERS_CLEAN",
    "Quarantine": "SELECT COUNT(*) FROM FRESHCART_DB.SILVER.ORDERS_QUARANTINE",
    "Deduplicated": "SELECT COUNT(*) FROM FRESHCART_DB.SILVER.ORDERS_DEDUP",
    "Gold rows and revenue": "SELECT COUNT(*), SUM(AMOUNT_INR) FROM FRESHCART_DB.GOLD.FCT_ORDERS",
}
for label, sql in verification_queries.items():
    cur.execute(sql)
    print(label, cur.fetchall())


## 5. Recorded results from the user's execution evidence
- Bronze raw rows: 8
- Silver checked: 8; clean: 6; quarantine: 2
- Quarantine reasons: `missing_customer_id` (ORD-1004), `unsupported_currency` (ORD-1006, EUR)
- Deduplicated orders: 4
- Gold fact rows: 4; revenue: ₹7,785; loyalty points: 126
- Six quality checks: zero problems after cleanup

These are recorded execution results, not outputs automatically produced by opening this notebook. Run the SQL against your Snowflake account to reproduce them.


In [ ]:
cur.close()
conn.close()
print("Snowflake connection closed.")
